# Generate water price and availability timeseries

- Water availability should be given as a variation, so each data point represents the variation with respect to the previous one.
- We should start with an initial value, and then based on the precipitation add, and substract some amount due to evaporation.
- The evaporation rate probably should be a linear function of the weather conditions (HR, Tamb): $\dot{V}_{loss} = a \cdot HR + b \cdot T_{amb} + c, \in [\underline{\dot{V}_{loss}}, \overline{\dot{V}_{loss}}]$
- To simplify the problem, assume a constat loss rate
- The relationship precipitation and water availability is arbitrary. To start with something, model it so that during winter the wet cooling tower could operate exclusively by using this water source, while on summer it would not suffice.


[Data source](https://collab.psa.es/apps/files/files/1267204?dir=/E47/E47_TECHNICAL/DESAL%20PROJECTS/1K235_SOLHYCOOL/Ejecuci%C3%B3n/T2.2%20-%20Optimizaci%C3%B3n%20de%20la%20operaci%C3%B3n&openfile=true)

In [1]:
from pathlib import Path
from loguru import logger
import datetime
import numpy as np
import pandas as pd

# Constants
data_path: Path = Path("../../data/datasets")
output_path: Path = Path("../results")

water_price_egypt = 0.06047057835 # €/m³
water_price_morocco = 0.0290538951 # €/m³

# Parameters
year_start: int = 2022
year_end: int = 2024

Vavail0: float = 20. # Period initial volume, m³
evap_rate: float = -0.5 # Evaporation rate, m³/h
alternative_source_cost_multiplier = 20.0 # Relative cost increase factor of source 2 with respect to source 1, -



In [2]:
# Create a pandas series with one-hour resolution
start_date = datetime.datetime(year_start, 1, 1, 0, 0, 0)
end_date = datetime.datetime(year_end, 12, 31, 23, 0, 0)
date_rng = pd.date_range(start=start_date, end=end_date, freq='h', tz='UTC')
fname: str = f"water_data_{start_date:%Y%m%d}_{end_date:%Y%m%d}"

# Initialize dataframe
df = pd.DataFrame(
    data={
        "water_price_morocco_eur_m3": water_price_morocco,
        "water_price_morocco_alternative_eur_m3": water_price_morocco * alternative_source_cost_multiplier,
        "water_price_egypt_eur_m3": water_price_egypt,
        "water_price_egypt_alternative_eur_m3": water_price_egypt * alternative_source_cost_multiplier,
    },
    index=date_rng,
)

df


,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3
2022-01-01 00:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2022-01-01 01:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2022-01-01 02:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2022-01-01 03:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2022-01-01 04:00:00+00:00,0.029054,0.581078,0.060471,1.209412
...,...,...,...,...
2024-12-31 19:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2024-12-31 20:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2024-12-31 21:00:00+00:00,0.029054,0.581078,0.060471,1.209412
2024-12-31 22:00:00+00:00,0.029054,0.581078,0.060471,1.209412


## Model water availability as a function of precipitation data

In [3]:
# For now just assign some initial value at the start of the timeseries,
# and then increase/decrease it by the same amount every day

# deltaV: hourly volume variation (m³/h)
# Vavail: available water volume (m³)

# Terribly inefficient, but whatever it's only done once
current_month = start_date.month
for single_date in pd.date_range(start=start_date, end=end_date, freq='h', tz='UTC'):
    if single_date.month != current_month:
        current_month = single_date.month
        Vavail = Vavail0 # Reset available volume every month
    deltaV = evap_rate + np.random.uniform(0.8*evap_rate, 1.2*evap_rate)
    Vavail = Vavail0 + deltaV
    df.loc[single_date, "Vavail_m3"] = Vavail
    df.loc[single_date, "deltaV_m3_h"] = deltaV
    df.loc[single_date, "Vavail_l"] = Vavail*1e3
    df.loc[single_date, "deltaV_l_h"] = deltaV*1e3

df


,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.951323,-1.048677,18951.323166,-1048.676834
2022-01-01 01:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.051194,-0.948806,19051.194432,-948.805568
2022-01-01 02:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.956947,-1.043053,18956.947372,-1043.052628
2022-01-01 03:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.065861,-0.934139,19065.861259,-934.138741
2022-01-01 04:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.947961,-1.052039,18947.960580,-1052.039420
...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.033793,-0.966207,19033.792916,-966.207084
2024-12-31 20:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.022591,-0.977409,19022.591357,-977.408643
2024-12-31 21:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.929140,-1.070860,18929.139769,-1070.860231
2024-12-31 22:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.976059,-1.023941,18976.058894,-1023.941106


In [4]:
# Export dataframe including additional units (liters)
df.to_hdf(output_path / f"{fname}.h5", key="data", mode="w", format="table")
df.to_csv(output_path / f"{fname}.csv")


In [5]:
df = pd.read_hdf(output_path / f"{fname}.h5", index_col=0, parse_dates=True)
df


,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,Vavail_m3,deltaV_m3_h,Vavail_l,deltaV_l_h
2022-01-01 00:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.951323,-1.048677,18951.323166,-1048.676834
2022-01-01 01:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.051194,-0.948806,19051.194432,-948.805568
2022-01-01 02:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.956947,-1.043053,18956.947372,-1043.052628
2022-01-01 03:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.065861,-0.934139,19065.861259,-934.138741
2022-01-01 04:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.947961,-1.052039,18947.960580,-1052.039420
...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.033793,-0.966207,19033.792916,-966.207084
2024-12-31 20:00:00+00:00,0.029054,0.581078,0.060471,1.209412,19.022591,-0.977409,19022.591357,-977.408643
2024-12-31 21:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.929140,-1.070860,18929.139769,-1070.860231
2024-12-31 22:00:00+00:00,0.029054,0.581078,0.060471,1.209412,18.976059,-1.023941,18976.058894,-1023.941106


In [6]:
# Visualize data

from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

var_ids: list[str] = [
    ["water_price_morocco_eur_m3", "water_price_egypt_eur_m3", "water_price_morocco_alternative_eur_m3", "water_price_egypt_alternative_eur_m3"], 
    "Vavail_m3",
]
var_units: list[str] = ["€/m<sup>3</sup>","m<sup>3</sup>"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    var_id = [var_id] if not isinstance(var_id, list) else var_id
    
    for v_id in var_id:
        fig.add_trace(
            go.Scattergl(name=v_id, showlegend=True), 
            hf_x=df.index, 
            hf_y=np.ascontiguousarray( df[v_id] ), 
            # max_n_samples=2_000,
            row=i + 1, col=1
        )
    fig.update_yaxes(title_text=var_unit, row=i + 1)

fig.update_layout(
    height=1000,
    width=800,
    title="<b>Environment variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.15, xanchor="left", x=0),
    margin=dict(l=20, r=20, t=250, b=20),
)

fig


FigureWidgetResampler({
    'data': [{'name': ('<b style="color:sandybrown">[R' ... ' style="color:#fc9944">~1D</i>'),
              'showlegend': True,
              'type': 'scattergl',
              'uid': '25f285ce-c6c1-4045-b16f-ac90b242a268',
              'x': array([datetime.datetime(2022, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 1, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 2, 4, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2024, 12, 29, 5, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 30, 7, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([0.0290539, 0.0290539, 0.0290539, ..., 0.0290539, 0.0290539, 0.0290539],

In [9]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range
save_figure(fig, f"solhycool_water_viz_{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", 
            figure_path=output_path, formats=["png", "svg"])


2025-02-28 14:26:04.283 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results')]/solhycool_water_viz_20221124_20221228.png
2025-02-28 14:26:07.805 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results')]/solhycool_water_viz_20221124_20221228.svg
